In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pengaturan default dan opsi konfigurasi transpilasi


<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami sarankan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Circuit abstrak perlu ditranspilasi karena QPU hanya mendukung sekumpulan gate basis tertentu dan tidak bisa menjalankan operasi sembarang. Fungsi Transpiler adalah mengubah circuit sembarang agar bisa berjalan di QPU tertentu. Ini dilakukan dengan menerjemahkan circuit ke gate basis yang didukung, dan menambahkan gate SWAP sesuai kebutuhan agar konektivitas circuit sesuai dengan QPU.

Seperti yang dijelaskan di [Transpile dengan pass manager](transpile-with-pass-managers), kamu bisa membuat [pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) menggunakan fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) dan mengoper circuit atau daftar circuit ke metode [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run)-nya untuk mentranspilasi. Kamu bisa memanggil `generate_preset_pass_manager` hanya dengan optimization level dan Backend, menggunakan nilai default untuk semua opsi lainnya, atau mengoper argumen tambahan ke fungsi tersebut untuk menyetel transpilasi secara lebih detail.

## Penggunaan dasar tanpa parameter
Pada contoh ini, kita mengoper circuit dan QPU target ke Transpiler tanpa menentukan parameter tambahan apa pun.

Buat circuit dan lihat hasilnya:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpilasi circuit dan lihat hasilnya:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Semua parameter yang tersedia
Berikut adalah semua parameter yang tersedia untuk fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). Ada dua kelompok argumen: yang mendeskripsikan target kompilasi, dan yang memengaruhi cara kerja Transpiler.

Semua parameter kecuali `optimization_level` bersifat opsional. Untuk detail lengkapnya, lihat [dokumentasi API Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - Seberapa banyak optimasi yang dilakukan pada circuit. Integer dalam rentang (0 - 3). Level yang lebih tinggi menghasilkan circuit yang lebih optimal, dengan imbalan waktu transpilasi yang lebih lama. Lihat [Atur level optimasi Transpiler](set-optimization) untuk detail lebih lanjut.

### Parameter untuk mendeskripsikan target kompilasi:
Argumen-argumen ini mendeskripsikan QPU target untuk eksekusi circuit, termasuk informasi seperti coupling map QPU (yang menggambarkan konektivitas Qubit), gate basis yang didukung QPU, dan tingkat error gate.

Banyak dari parameter ini dijelaskan secara detail di [Parameter yang umum digunakan untuk transpilasi](common-parameters).

<details>
  <summary>
    **Parameter QPU (`Backend`)**
  </summary>

**Parameter Backend** - Jika kamu menentukan `backend`, kamu tidak perlu menentukan `target` atau opsi Backend lainnya. Begitu pula, jika kamu menentukan `target`, kamu tidak perlu menentukan `backend` atau opsi Backend lainnya.
- `backend` (Backend) - Jika ini diatur, Transpiler mengompilasi circuit input ke perangkat ini. Jika ada opsi lain yang memengaruhi pengaturan ini, seperti `coupling_map`, maka opsi tersebut menggantikan pengaturan dari `backend`.
- `target` (Target) - Target Transpiler Backend. Biasanya ini ditentukan sebagai bagian dari argumen Backend, tapi jika kamu membuat objek Target secara manual, kamu bisa menentukannya di sini. Ini menggantikan target dari `backend`.
- `backend_properties` (BackendProperties) - Properti yang dikembalikan oleh QPU, termasuk informasi tentang error gate, error readout, waktu koherensi Qubit, dan sebagainya. Temukan QPU yang menyediakan informasi ini dengan menjalankan `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - Batasan opsional pada resolusi waktu instruksi dari hardware kontrol. Informasi ini disediakan oleh konfigurasi QPU. Jika QPU tidak memiliki batasan pada alokasi waktu instruksi, `timing_constraints` bernilai `None` dan tidak ada penyesuaian yang dilakukan. Sebuah QPU bisa melaporkan sekumpulan batasan, yaitu:
    - `granularity`: Nilai integer yang mewakili resolusi minimum gate pulsa dalam satuan dt. Gate pulsa yang ditentukan pengguna harus memiliki durasi yang merupakan kelipatan dari nilai granularitas ini.
    - `min_length`: Nilai integer yang mewakili panjang minimum gate pulsa dalam satuan dt. Gate pulsa yang ditentukan pengguna harus lebih panjang dari nilai ini.
    - `pulse_alignment`: Nilai integer yang mewakili resolusi waktu dari waktu mulai instruksi gate. Instruksi gate harus dimulai pada waktu yang merupakan kelipatan dari nilai ini.
    - `acquire_alignment`: Nilai integer yang mewakili resolusi waktu dari waktu mulai instruksi measure. Instruksi measure harus dimulai pada waktu yang merupakan kelipatan dari nilai ini.
</details>

<details>
  <summary>
    **Parameter layout dan topologi**
  </summary>

- `basis_gates` (List[str] | None) - Daftar nama gate basis untuk di-unroll. Misalnya ['u1', 'u2', 'u3', 'cx']. Jika `None`, tidak dilakukan unroll.
- `coupling_map` (CouplingMap | List[List[int]]) - Coupling map berarah (bisa kustom) sebagai target dalam pemetaan. Jika coupling map simetris, kedua arah perlu ditentukan. Format yang didukung:
    - Instance CouplingMap
    - List - harus diberikan sebagai matriks ketetanggaan, di mana setiap entri menentukan semua interaksi dua Qubit berarah yang didukung QPU. Misalnya: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Pemetaan operasi circuit ke jadwal pulsa. Jika `None`, `instruction_schedule_map` QPU yang digunakan.
</details>

### Parameter untuk memengaruhi cara kerja Transpiler
Parameter-parameter ini memengaruhi tahap transpilasi tertentu. Beberapa di antaranya bisa memengaruhi beberapa tahap, tapi hanya dicantumkan di satu tahap untuk menyederhanakan. Jika kamu menentukan argumen, seperti `initial_layout` untuk Qubit yang ingin digunakan, nilai tersebut menggantikan semua pass yang bisa mengubahnya. Dengan kata lain, Transpiler tidak akan mengubah apa pun yang kamu tentukan secara manual. Untuk detail tentang tahap-tahap tertentu, lihat [Tahap Transpiler](transpiler-stages).

<details>
  <summary>
    **Tahap inisialisasi**
  </summary>

- `hls_config` (HLSConfig) - Kelas konfigurasi opsional `HLSConfig` yang dioper langsung ke pass transformasi `HighLevelSynthesis`. Kelas konfigurasi ini memungkinkan kamu menentukan daftar algoritma sintesis dan parameternya untuk berbagai objek tingkat tinggi.
- `init_method` (str) - Nama plugin yang digunakan untuk tahap inisialisasi. Secara default, plugin eksternal tidak digunakan. Kamu bisa melihat daftar plugin yang terinstal dengan menjalankan `list_stage_plugins()` dengan `init` sebagai argumen nama tahap.
- `unitary_synthesis_method` (str) - Nama metode sintesis unitary yang digunakan. Secara default, `default` digunakan. Kamu bisa melihat daftar plugin yang terinstal dengan menjalankan `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - Dictionary konfigurasi opsional yang dioper langsung ke plugin sintesis unitary. Secara default pengaturan ini tidak berpengaruh karena metode sintesis unitary default tidak menggunakan konfigurasi kustom. Menerapkan konfigurasi kustom hanya diperlukan ketika plugin sintesis unitary ditentukan dengan argumen `unitary_synthesis`. Karena ini bersifat kustom untuk setiap plugin sintesis unitary, lihat dokumentasi plugin untuk cara menggunakan opsi ini.
</details>

<details>
  <summary>
    **Tahap layout**
  </summary>

- `initial_layout` (Layout | Dict | List) - Posisi awal Qubit virtual pada Qubit fisik. Jika layout ini membuat circuit kompatibel dengan batasan `coupling_map`, layout ini akan digunakan. Layout akhir tidak dijamin sama, karena Transpiler bisa memindahkan Qubit melalui swap atau cara lain. Untuk detail lengkapnya, lihat [bagian Initial layout.](common-parameters#initial-layout)
- `layout_method` (str) - Nama pass pemilihan layout (`default`, `dense`, `sabre`, dan `trivial`). Ini juga bisa berupa nama plugin eksternal yang digunakan untuk tahap layout. Kamu bisa melihat daftar plugin yang terinstal dengan menjalankan `list_stage_plugins()` dengan `layout` untuk argumen `stage_name`. Nilai defaultnya adalah `sabre`.
</details>

<details>
  <summary>
    **Tahap routing**
  </summary>

- `routing_method` (str) - Nama pass routing (`basic`, `lookahead`, `default`, `sabre`, atau `none`). Ini juga bisa berupa nama plugin eksternal yang digunakan untuk tahap routing. Kamu bisa melihat daftar plugin yang terinstal dengan menjalankan `list_stage_plugins()` dengan `routing` untuk argumen `stage_name`. Nilai defaultnya adalah `sabre`.
</details>

<details>
  <summary>
    **Tahap translation**
  </summary>

- `translation_method` (str) - Nama pass translation (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Ini juga bisa berupa nama plugin eksternal yang digunakan untuk tahap translation. Kamu bisa melihat daftar plugin yang terinstal dengan menjalankan `list_stage_plugins()` dengan `translation` untuk argumen `stage_name`. Nilai defaultnya adalah `translator`.
</details>

<details>
  <summary>
    **Tahap optimasi**
  </summary>

- `approximation_degree` (float, dalam rentang 0-1 | None) - Dial heuristik yang digunakan untuk aproksimasi circuit (1.0 = tidak ada aproksimasi, 0.0 = aproksimasi maksimal). Nilai defaultnya adalah 1.0. Menentukan `None` mengatur derajat aproksimasi ke tingkat error yang dilaporkan. Lihat [bagian Approximation degree](common-parameters#approx-degree) untuk detail lebih lanjut.
- `optimization_method` (str) - Nama plugin yang digunakan untuk tahap optimasi. Secara default plugin eksternal tidak digunakan. Kamu bisa melihat daftar plugin yang terinstal dengan menjalankan `list_stage_plugins()` dengan `optimization` untuk argumen `stage_name`.
</details>

<details>
  <summary>
    **Tahap scheduling**
  </summary>

- `scheduling_method` (str) - Nama pass scheduling. Ini juga bisa berupa nama plugin eksternal yang digunakan untuk tahap scheduling. Kamu bisa melihat daftar plugin yang terinstal dengan menjalankan `list_stage_plugins()` dengan `scheduling` untuk argumen `stage_name`.
  * 'as_soon_as_possible': Jadwalkan instruksi secara greedy, sesegera mungkin pada resource Qubit (alias: `asap`).
  * 'as_late_as_possible': Jadwalkan instruksi selambat mungkin, yaitu menjaga Qubit dalam ground state jika memungkinkan (alias: `alap`). Ini adalah nilai default.
</details>

<details>
  <summary>
    **Eksekusi Transpiler**
  </summary>

- `seed_transpiler` (int) - Mengatur seed acak untuk bagian stokastik dari Transpiler.
</details>

Nilai default berikut digunakan jika kamu tidak menentukan parameter apa pun di atas. Lihat [halaman referensi API](../api/qiskit/transpiler_preset) metode ini untuk informasi lebih lanjut:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>